# Evaluación de los Sistemas de Recuperación

Se realiza la evaluación de rendimiento de los dos sistemas de búsqueda implementados:
1. **Sistema de Recuperación Booleano** (evaluado usando las métricas de **Precisión** y **Sensibilidad (Recall)** promediadas).
2. **Sistema de Recuperación Vectorial** (TF-IDF + Coseno, evaluado usando **MAP (Mean Average Precision)**).

Utiliza las 20 necesidades de información y consultas guardadas en `consultas.csv` junto con sus juicios de relevancia correspondientes de `relevancia.csv`.

In [1]:
import pandas as pd
import numpy as np
from whoosh.index import open_dir
from funciones import (
    obtener_resultados_booleanos,
    obtener_resultados_coseno
)

In [2]:
df_consultas = pd.read_csv("consultas.csv")
df_relevancia = pd.read_csv("relevancia.csv")

ix = open_dir("indexdir")
with ix.searcher() as searcher:
    documentos = list(searcher.all_stored_fields())

resultados_eval = []

# Bucle para evaluar cada consulta
for index, row in df_consultas.iterrows():
    need_id = int(row['necesidad_id'])
    query_bool = row['consulta_booleana']
    query_text = row['consulta_texto_libre']
    
    # Relevantes reales de la consulta actual
    relevantes = set(df_relevancia[(df_relevancia['necesidad_id'] == need_id) & (df_relevancia['relevante'] == 1)]['documento_id'].tolist())
    
    # Evaluacion Booleana (Precision y Recall)
    recuperados_bool = obtener_resultados_booleanos(query_bool)
    if len(recuperados_bool) > 0:
        precision = len(recuperados_bool.intersection(relevantes)) / len(recuperados_bool)
    else:
        precision = 0.0
        
    if len(relevantes) > 0:
        recall = len(recuperados_bool.intersection(relevantes)) / len(relevantes)
    else:
        recall = 0.0
        
    # Evaluacion Coseno (AP)
    recuperados_cos = obtener_resultados_coseno(query_text, documentos)
    hits = 0
    sum_precisions = 0.0
    for k, doc_id in enumerate(recuperados_cos, 1):
        if doc_id in relevantes:
            hits += 1
            sum_precisions += hits / k
            
    if len(relevantes) > 0:
        ap = sum_precisions / len(relevantes)
    else:
        ap = 0.0
        
    resultados_eval.append({
        "Necesidad": need_id,
        "Precision Booleana": precision,
        "Recall Booleano": recall,
        "AP Coseno": ap
    })
    
df_eval = pd.DataFrame(resultados_eval)

mean_prec = df_eval['Precision Booleana'].mean()
mean_rec = df_eval['Recall Booleano'].mean()
map_coseno = df_eval['AP Coseno'].mean()

print("Precision Booleana Promedio:", round(mean_prec, 4))
print("Recall Booleano Promedio:", round(mean_rec, 4))
print("MAP Coseno:", round(map_coseno, 4))
print()

df_eval.round(4)

Precision Booleana Promedio: 0.998
Recall Booleano Promedio: 1.0
MAP Coseno: 0.7248



,Necesidad,Precision Booleana,Recall Booleano,AP Coseno
0,1,1.00,1.0,0.7649
1,2,1.00,1.0,0.4289
2,3,1.00,1.0,0.8752
3,4,1.00,1.0,0.2766
4,5,1.00,1.0,0.6679
5,6,0.96,1.0,0.6075
6,7,1.00,1.0,1.0000
7,8,1.00,1.0,1.0000
8,9,1.00,1.0,1.0000
9,10,1.00,1.0,0.5429
